# DLL Runtime Test: TFLiteGridSample (Built-In Registration)

This notebook builds a minimal `.tflite` custom-op model and executes it with `tensorflowlite_c.dll` where `TFLiteGridSample` is compiled and registered in the default resolver.

In [ ]:
from pathlib import Path
import os
import ctypes

import numpy as np
import tensorflow as tf
from tensorflow.lite.python import schema_py_generated as schema_fb
from tensorflow.lite.tools import flatbuffer_utils

start = Path.cwd().resolve()
CUSTOM_OPS_ROOT = None
for candidate in [start, *start.parents]:
    if (candidate / "CMakeLists.txt").exists() and (candidate / "grid_sample.cc").exists():
        CUSTOM_OPS_ROOT = candidate
        break
    if (candidate / "custom_ops" / "CMakeLists.txt").exists():
        CUSTOM_OPS_ROOT = candidate / "custom_ops"
        break

if CUSTOM_OPS_ROOT is None:
    raise RuntimeError("Could not locate custom_ops root containing CMakeLists.txt and grid_sample.cc.")

ARTIFACT_DIR = CUSTOM_OPS_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RAW_MODEL = ARTIFACT_DIR / "grid_sample_pyfunc_raw.tflite"
PATCHED_MODEL = ARTIFACT_DIR / "grid_sample_custom.tflite"

TFLITE_C_DLL_CANDIDATES = [
    CUSTOM_OPS_ROOT / "build" / "Release" / "tensorflowlite_c.dll",
    CUSTOM_OPS_ROOT / "build" / "_deps" / "tensorflow-lite-build" / "c" / "Release" / "tensorflowlite_c.dll",
]
TFLITE_C_DLL = next((p for p in TFLITE_C_DLL_CANDIDATES if p.exists()), TFLITE_C_DLL_CANDIDATES[0])

print("CUSTOM_OPS_ROOT:", CUSTOM_OPS_ROOT)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("RAW_MODEL:", RAW_MODEL)
print("PATCHED_MODEL:", PATCHED_MODEL)
print("TFLITE_C_DLL:", TFLITE_C_DLL, "exists:", TFLITE_C_DLL.exists())

In [ ]:
# Build a TFLite model containing a PyFunc op which we
# will later patch to be our custom op.

class ExportAsPyFunc(tf.keras.layers.Layer):
    def call(self, inputs):
        images, theta = inputs
        out = tf.raw_ops.PyFunc(
            input=[images, theta],
            token="TFLiteGridSample",
            Tout=[tf.float32],
            name="TFLiteGridSample",
        )
        if isinstance(out, (list, tuple)):
            out = out[0]
        out.set_shape(images.shape)
        return out

image_in = tf.keras.Input(shape=(4, 4, 1), batch_size=1, dtype=tf.float32, name="image")
theta_in = tf.keras.Input(shape=(6,), batch_size=1, dtype=tf.float32, name="theta")
model = tf.keras.Model(
    inputs=[image_in, theta_in],
    outputs=ExportAsPyFunc()([image_in, theta_in]),
)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.allow_custom_ops = True
RAW_MODEL.write_bytes(converter.convert())

print("Wrote raw model:", RAW_MODEL)

In [ ]:
# Patch the model to replace the PyFunc operator code with our
# custom op code via flatbuffer_utils, which is a thin wrapper
# around the flatbuffers Python API.

model_obj = flatbuffer_utils.read_model(str(RAW_MODEL))
patched_count = 0
for op_code in model_obj.operatorCodes:
    if op_code.customCode in (b"PyFunc", "PyFunc"):
        op_code.customCode = b"TFLiteGridSample"
        patched_count += 1

if patched_count == 0:
    raise RuntimeError("No PyFunc operator code found to patch.")

flatbuffer_utils.write_model(model_obj, str(PATCHED_MODEL))
print("Patched operator code count:", patched_count)
print("Wrote patched model:", PATCHED_MODEL)

m = schema_fb.Model.GetRootAsModel(PATCHED_MODEL.read_bytes(), 0)
for i in range(m.OperatorCodesLength()):
    oc = m.OperatorCodes(i)
    print("OperatorCode", i, "builtin", oc.BuiltinCode(), "custom", oc.CustomCode())

In [ ]:
# Attempt to load the patched model with the stock TFLite interpreter, which should
# fail.

# If it does not fail, that means the stock interpreter is somehow able to resolve
# the custom op, which would be unexpected and could mask issues with our patched
# model or custom op registration.

try:
    interp = tf.lite.Interpreter(model_path=str(PATCHED_MODEL))
    interp.allocate_tensors()
    print("Unexpected: stock tf.lite.Interpreter allocated custom model.")
except Exception as exc:
    print("Expected failure in stock interpreter:", type(exc).__name__)
    print(str(exc).splitlines()[0])

In [ ]:
# Now load the patched model with the C API, which should succeed since
# our custom op registration should be visible to it.

# We will also run inference to verify the custom op is working as expected,
# which will confirm that the patched model is valid and that the custom op
# registration is working correctly.

if not TFLITE_C_DLL.exists():
    raise FileNotFoundError(f"Missing tensorflowlite_c.dll: {TFLITE_C_DLL}")

os.add_dll_directory(str(TFLITE_C_DLL.parent))
lib = ctypes.CDLL(str(TFLITE_C_DLL))

c_void_p = ctypes.c_void_p
c_int = ctypes.c_int
c_size_t = ctypes.c_size_t
kTfLiteOk = 0

lib.TfLiteVersion.restype = ctypes.c_char_p
lib.TfLiteModelCreateFromFile.argtypes = [ctypes.c_char_p]
lib.TfLiteModelCreateFromFile.restype = c_void_p
lib.TfLiteModelDelete.argtypes = [c_void_p]

lib.TfLiteInterpreterOptionsCreate.restype = c_void_p
lib.TfLiteInterpreterOptionsDelete.argtypes = [c_void_p]
lib.TfLiteInterpreterOptionsSetNumThreads.argtypes = [c_void_p, c_int]

lib.TfLiteInterpreterCreate.argtypes = [c_void_p, c_void_p]
lib.TfLiteInterpreterCreate.restype = c_void_p
lib.TfLiteInterpreterDelete.argtypes = [c_void_p]

lib.TfLiteInterpreterAllocateTensors.argtypes = [c_void_p]
lib.TfLiteInterpreterAllocateTensors.restype = c_int
lib.TfLiteInterpreterInvoke.argtypes = [c_void_p]
lib.TfLiteInterpreterInvoke.restype = c_int

lib.TfLiteInterpreterGetInputTensorCount.argtypes = [c_void_p]
lib.TfLiteInterpreterGetInputTensorCount.restype = c_int
lib.TfLiteInterpreterGetOutputTensorCount.argtypes = [c_void_p]
lib.TfLiteInterpreterGetOutputTensorCount.restype = c_int
lib.TfLiteInterpreterGetInputTensor.argtypes = [c_void_p, c_int]
lib.TfLiteInterpreterGetInputTensor.restype = c_void_p
lib.TfLiteInterpreterGetOutputTensor.argtypes = [c_void_p, c_int]
lib.TfLiteInterpreterGetOutputTensor.restype = c_void_p

lib.TfLiteTensorNumDims.argtypes = [c_void_p]
lib.TfLiteTensorNumDims.restype = c_int
lib.TfLiteTensorDim.argtypes = [c_void_p, c_int]
lib.TfLiteTensorDim.restype = c_int
lib.TfLiteTensorByteSize.argtypes = [c_void_p]
lib.TfLiteTensorByteSize.restype = c_size_t
lib.TfLiteTensorCopyFromBuffer.argtypes = [c_void_p, c_void_p, c_size_t]
lib.TfLiteTensorCopyFromBuffer.restype = c_int
lib.TfLiteTensorCopyToBuffer.argtypes = [c_void_p, c_void_p, c_size_t]
lib.TfLiteTensorCopyToBuffer.restype = c_int

def tensor_shape(t):
    return [lib.TfLiteTensorDim(t, d) for d in range(lib.TfLiteTensorNumDims(t))]

model_handle = lib.TfLiteModelCreateFromFile(str(PATCHED_MODEL).encode("utf-8"))
if not model_handle:
    raise RuntimeError("TfLiteModelCreateFromFile failed")

options = lib.TfLiteInterpreterOptionsCreate()
lib.TfLiteInterpreterOptionsSetNumThreads(options, 1)
interpreter = lib.TfLiteInterpreterCreate(model_handle, options)
if not interpreter:
    raise RuntimeError("TfLiteInterpreterCreate failed")

try:
    status = lib.TfLiteInterpreterAllocateTensors(interpreter)
    if status != kTfLiteOk:
        raise RuntimeError(f"Allocate failed with status {status}")

    n_in = lib.TfLiteInterpreterGetInputTensorCount(interpreter)
    n_out = lib.TfLiteInterpreterGetOutputTensorCount(interpreter)
    print("TfLiteVersion:", lib.TfLiteVersion().decode("utf-8"))
    print("Input tensors:", n_in, "Output tensors:", n_out)

    image = np.arange(1, 17, dtype=np.float32).reshape(1, 4, 4, 1) / 16.0
    theta_identity = np.array([[1, 0, 0, 0, 1, 0]], dtype=np.float32)

    image_idx = None
    theta_idx = None
    for i in range(n_in):
        t = lib.TfLiteInterpreterGetInputTensor(interpreter, i)
        shape = tensor_shape(t)
        print("input", i, "shape", shape)
        if shape == [1, 4, 4, 1]:
            image_idx = i
        elif shape == [1, 6]:
            theta_idx = i

    if image_idx is None or theta_idx is None:
        raise RuntimeError("Could not find expected image/theta input tensors.")

    t_image = lib.TfLiteInterpreterGetInputTensor(interpreter, image_idx)
    t_theta = lib.TfLiteInterpreterGetInputTensor(interpreter, theta_idx)

    status = lib.TfLiteTensorCopyFromBuffer(t_image, image.ctypes.data_as(c_void_p), image.nbytes)
    if status != kTfLiteOk:
        raise RuntimeError(f"Image copy failed with status {status}")

    status = lib.TfLiteTensorCopyFromBuffer(t_theta, theta_identity.ctypes.data_as(c_void_p), theta_identity.nbytes)
    if status != kTfLiteOk:
        raise RuntimeError(f"Theta copy failed with status {status}")

    status = lib.TfLiteInterpreterInvoke(interpreter)
    if status != kTfLiteOk:
        raise RuntimeError(f"Invoke failed with status {status}")

    t_out = lib.TfLiteInterpreterGetOutputTensor(interpreter, 0)
    out_shape = tensor_shape(t_out)
    out_nbytes = lib.TfLiteTensorByteSize(t_out)
    output = np.empty(out_nbytes // 4, dtype=np.float32)

    status = lib.TfLiteTensorCopyToBuffer(t_out, output.ctypes.data_as(c_void_p), out_nbytes)
    if status != kTfLiteOk:
        raise RuntimeError(f"Output copy failed with status {status}")

    output = output.reshape(out_shape)
    max_abs_diff = float(np.max(np.abs(output - image)))
    print("Output shape:", out_shape)
    print("Max abs diff vs identity:", max_abs_diff)
    assert max_abs_diff <= 1e-4, "Identity theta should preserve the image."

    print("SUCCESS: tensorflowlite_c.dll executed TFLiteGridSample via default resolver registration.")
finally:
    if interpreter:
        lib.TfLiteInterpreterDelete(interpreter)
    if options:
        lib.TfLiteInterpreterOptionsDelete(options)
    if model_handle:
        lib.TfLiteModelDelete(model_handle)